In [0]:
# Delta Lake Deep Dive
# The foundation of Databricks Lakehouse!

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
# 1. Create Delta table with schema
schema = StructType([
    StructField("id", IntegerType()),
    StructField("model", StringType()),
    StructField("accuracy", FloatType()),
    StructField("version", IntegerType()),
    StructField("timestamp", StringType())
])

data = [
    (1, "RF_v1", 88.0, 1, "2026-03-01"),
    (2, "XGB_v1", 92.0, 1, "2026-03-02"),
    (3, "NN_v1", 95.0, 1, "2026-03-03")
]
df = spark.createDataFrame(data, schema)

# Create managed Delta table in Unity Catalog
df.write.format("delta").mode("overwrite").saveAsTable(
    "ml_registry"
)
print("Delta table created!")

Delta table created!


In [0]:
# 2. MERGE — upsert operation (huge Databricks feature!)
delta_table = DeltaTable.forName(
    spark, "ml_registry"
)

new_data = spark.createDataFrame([
    (2, "XGB_v2", 94.5, 2, "2026-03-15"),  # update
    (4, "CNN_v1", 91.0, 1, "2026-03-15")   # insert
], schema)

# MERGE = UPDATE existing + INSERT new
delta_table.alias("old").merge(
    new_data.alias("new"),
    "old.id = new.id"
).whenMatchedUpdate(set={
    "accuracy": "new.accuracy",
    "version": "new.version",
    "timestamp": "new.timestamp"
}).whenNotMatchedInsert(values={
    "id": "new.id",
    "model": "new.model",
    "accuracy": "new.accuracy",
    "version": "new.version",
    "timestamp": "new.timestamp"
}).execute()

print("\nAfter MERGE:")
spark.read.table("ml_registry").orderBy("id").show()


After MERGE:
+---+------+--------+-------+----------+
| id| model|accuracy|version| timestamp|
+---+------+--------+-------+----------+
|  1| RF_v1|    88.0|      1|2026-03-01|
|  2|XGB_v1|    94.5|      2|2026-03-15|
|  3| NN_v1|    95.0|      1|2026-03-03|
|  4|CNN_v1|    91.0|      1|2026-03-15|
+---+------+--------+-------+----------+



In [0]:
# 3. Time travel — query any version!
print("\nVersion 0 (original):")
spark.read.format("delta").option(
    "versionAsOf", 0
).table("ml_registry").show()


Version 0 (original):
+---+------+--------+-------+----------+
| id| model|accuracy|version| timestamp|
+---+------+--------+-------+----------+
|  1| RF_v1|    88.0|      1|2026-03-01|
|  2|XGB_v1|    92.0|      1|2026-03-02|
|  3| NN_v1|    95.0|      1|2026-03-03|
+---+------+--------+-------+----------+



In [0]:
# 4. Delta table history
print("\nTable history:")
delta_table.history().select(
    "version", "timestamp", "operation"
).show()


Table history:
+-------+-------------------+--------------------+
|version|          timestamp|           operation|
+-------+-------------------+--------------------+
|      2|2026-03-30 13:10:51|            OPTIMIZE|
|      1|2026-03-30 13:10:47|               MERGE|
|      0|2026-03-30 13:09:40|CREATE OR REPLACE...|
+-------+-------------------+--------------------+



In [0]:
# 5. Schema evolution
spark.sql("""
    ALTER TABLE ml_registry
    ADD COLUMN f1_score FLOAT
""")
print("Schema evolved — added f1_score column!")

Schema evolved — added f1_score column!


In [0]:
# 6. Optimize + Z-Order (Spark cert topic!)
spark.sql("""
    OPTIMIZE ml_registry
    ZORDER BY (model)
""")
print("Table optimized with Z-Order!")

Table optimized with Z-Order!
